# 🔍 Interpretabilidad — Análisis SHAP

**Proyecto:** Predicción de calidad de crudos mediante Machine Learning  
**Autor:** Tomás Inti Malafiej — Licenciatura en Ciencia de Datos, UCASAL

---

## ¿Qué es SHAP?

**SHAP** (SHapley Additive exPlanations) es un framework de interpretabilidad basado en la teoría de juegos cooperativos. Asigna a cada feature un valor que representa su **contribución marginal** a la predicción de cada muestra.

Básicamnente explica el por qué de la decisión tomada y cuánto influyó cada variable. Es como el traductor de lo que está pensando el modelo.

## ¿Por qué es importante en este contexto?

En un entorno de **certificación industrial** los analistas necesitan entender *por qué* el sistema clasifica un crudo como agrio antes de tomar decisiones comerciales basadas en esa predicción.

SHAP permite:
- **Auditar** cada predicción con trazabilidad completa.
- **Validar** que el modelo aprendió relaciones físicamente coherentes.
- **Comunicar** resultados de manera clara.

## Modelo analizado

Aplicamos SHAP al **Gradient Boosting** (modelo principal, AUC-ROC 0.935) para la tarea de clasificación Dulce/Agrio.

## 1. Configuración y cálculo de valores SHAP

In [ ]:
import shap
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Cargar modelo y datos
gb        = joblib.load('../data/modelo_gb_dulce_agrio.pkl')
scaler_gb = joblib.load('../data/scaler_dulce_agrio.pkl')

df = pd.read_csv('../data/crude_oil_clean.csv')
df['DULCE_AGRIO'] = (df['SRC'] >= 0.5).astype(int)

FEATURES = [
    'API_CRUDE', 'CRN', 'SU100', 'POUR_POINT',
    'CAR_CR_WT', 'LT_GAS_VOL', 'GAS_NP_VOL', 'RESDUM_VOL'
]

NOMBRES = {
    'API_CRUDE':   'Gravedad API',
    'CRN':         'Nitrógeno (%)',
    'SU100':       'Viscosidad 100°F',
    'POUR_POINT':  'Punto de fluidez',
    'CAR_CR_WT':   'Residuo carbono (%)',
    'LT_GAS_VOL':  'Vol. gasolina ligera',
    'GAS_NP_VOL':  'Vol. gasolina/nafta',
    'RESDUM_VOL':  'Vol. residuo'
}

X = df[FEATURES].copy()
y = df['DULCE_AGRIO'].copy()

X_scaled = scaler_gb.transform(X)
_, X_test, _, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# DataFrame con nombres legibles para SHAP
X_test_df = pd.DataFrame(X_test, columns=[NOMBRES[f] for f in FEATURES])

print(f'Muestras de test: {len(X_test_df):,}')
print('Calculando valores SHAP...')

# Calcular SHAP — TreeExplainer es exacto y eficiente para modelos de árbol
explainer   = shap.TreeExplainer(gb)
shap_values = explainer.shap_values(X_test)

print('Listo.')

## 2. Importancia global — ¿Qué variables importan más?

### 2.1 Beeswarm plot

El gráfico de beeswarm muestra la distribución completa de los valores SHAP para todas las muestras del test set. Cada punto es una muestra.

- **Eje X:** valor SHAP (positivo = empuja hacia agrio, negativo = empuja hacia dulce)
- **Color:** valor real de la feature (rojo = alto, azul = bajo)
- **Orden:** las variables más importantes están arriba

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test_df,
    plot_type='dot',
    show=False
)
plt.title('Impacto de cada variable en la predicción — Gradient Boosting\n'
          '(rojo = empuja hacia agrio, azul = empuja hacia dulce)',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 1 guardado.')

**Interpretación del beeswarm:**

| Variable | Patrón observado | Coherencia física |
|---|---|---|
| Gravedad API | API alto (rojo) → dulce; API bajo (azul) → agrio | ✅ Los crudos livianos tienden a ser dulces |
| Residuo carbono | Residuo alto (rojo) → agrio fuertemente | ✅ Los crudos sucios tienen más azufre |
| Vol. gasolina/nafta | Vol. alto (rojo) → dulce | ✅ Alta nafta indica crudo liviano |
| Nitrógeno | Nitrógeno alto → agrio moderado | ✅ N y S ocurren en crudos pesados |
| Punto de fluidez | Impacto mínimo y mixto | ✅ Baja correlación con azufre (EDA) |

### 2.2 Importancia media (barras)

El gráfico de barras muestra el valor absoluto medio de los SHAP values — una medida resumida de cuánto contribuye cada variable en promedio.

In [ ]:
plt.figure(figsize=(9, 5))
shap.summary_plot(
    shap_values, X_test_df,
    plot_type='bar',
    show=False
)
plt.title('Importancia media de variables (|SHAP|)\n'
          'Gradient Boosting — Predicción Dulce/Agrio',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/shap_importancia_barras.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 2 guardado.')

## 3. Explicación individual — ¿Por qué el modelo decidió esto?

Seleccionamos dos casos reales del test set con alta confianza para ilustrar cómo SHAP explica predicciones individuales:

- **Caso AGRIO:** muestra real clasificada como agria con confianza > 85%
- **Caso DULCE:** muestra real clasificada como dulce con confianza > 85%

Los valores grises en cada barra son los **valores reales de la feature** para esa muestra.

In [ ]:
probas    = gb.predict_proba(X_test)[:, 1]
idx_agrio = np.where((y_test.values == 1) & (probas > 0.85))[0][0]
idx_dulce = np.where((y_test.values == 0) & (probas < 0.15))[0][0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Explicación de predicciones individuales — SHAP Force Plot',
             fontsize=13, fontweight='bold')

for ax, idx, titulo in [
    (axes[0], idx_agrio, f'Caso real: AGRIO (confianza: {probas[idx_agrio]*100:.1f}%)'),
    (axes[1], idx_dulce, f'Caso real: DULCE (confianza: {(1-probas[idx_dulce])*100:.1f}%)')
]:
    valores_shap = shap_values[idx]
    valores_feat = X_test_df.iloc[idx]
    nombres_feat = X_test_df.columns.tolist()

    orden   = np.argsort(np.abs(valores_shap))[::-1][:6]
    colores = ['#e74c3c' if v > 0 else '#3498db' for v in valores_shap[orden]]
    bars = ax.barh(
        [nombres_feat[i] for i in orden][::-1],
        valores_shap[orden][::-1],
        color=colores[::-1], edgecolor='white'
    )
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_title(titulo, fontweight='bold', fontsize=11)
    ax.set_xlabel('Contribución SHAP\n(+ → agrio, − → dulce)')

    for bar, i in zip(bars, orden[::-1]):
        val_real = valores_feat[nombres_feat[i]]
        ax.text(
            bar.get_width() + 0.02 if bar.get_width() >= 0 else bar.get_width() - 0.02,
            bar.get_y() + bar.get_height() / 2,
            f'{val_real:.2f}',
            va='center',
            ha='left' if bar.get_width() >= 0 else 'right',
            fontsize=8, color='gray'
        )

plt.tight_layout()
plt.savefig('../data/shap_casos_individuales.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 3 guardado.')

## 4. Conclusiones del análisis SHAP

### Hallazgos principales

**Gravedad API y Residuo de carbono** son los dos predictores dominantes con importancia SHAP media de ~1.26 y ~1.14 respectivamente. Juntos explican la mayor parte de las decisiones del modelo.

**Validación de coherencia física:**
- API alto → dulce ✅
- Residuo carbono alto → agrio ✅  
- Vol. gasolina/nafta alto → dulce ✅
- Nitrógeno alto → agrio moderado ✅

Todas las relaciones aprendidas por el modelo son **físicamente coherentes** con la química del petróleo, lo que valida que el modelo capturó patrones reales.

**Punto de fluidez y Viscosidad** tienen impacto mínimo, consistente con su baja correlación con el azufre observada en el EDA (notebook 01).

---
